# SlµArch API example

This notebook builds a short synthetic EEG recordings. Synthetic events illustrate API usage, real recordings require appropriate preprocessing and stage/artifact masks.

In [ ]:
import mne
import numpy as np

from sluarch import detect_kcomp,detect_slowosc,detect_spindles

rng=np.random.default_rng(42)
sf=256

## Sleep spindles

`detect_spindles` supports `method='auto'` for recording-specific band selection and `method='fixed'` for the configured slow and fast bands.

In [ ]:
duration=60
times=np.arange(int(duration*sf))/sf
spindle_signal=rng.normal(0,8,times.size)  # microvolts
for center,frequency in [(8,11),(22,14),(38,11),(52,14)]:
    window=np.abs(times-center)<=0.6
    envelope=np.sin(np.pi*(times[window]-center+0.6)/1.2)**2
    spindle_signal[window]+=45*envelope*np.sin(2*np.pi*frequency*times[window])
info=mne.create_info(['EEG'],sf,ch_types=['eeg'])
raw_spindles=mne.io.RawArray((spindle_signal*1e-6)[None,:],info,verbose=False)
auto_spindles=detect_spindles(raw_spindles,method='auto')
fixed_spindles=detect_spindles(raw_spindles,method='fixed')
print('Auto method:',auto_spindles['count'],'events')
print('Fixed method:',fixed_spindles['count'],'events')

## Slow oscillations

Slow oscillations are returned as candidate events with waveform and spectral measures.

In [ ]:
so_signal=110*np.sin(2*np.pi*0.8*times)+rng.normal(0,3,times.size)
raw_so=mne.io.RawArray((so_signal*1e-6)[None,:],info,verbose=False)
slow_oscillations=detect_slowosc(raw_so)
print('Slow oscillation candidates:',slow_oscillations['count'])
print('Spectral score:',slow_oscillations['spectral']['score'])

## K-complexes

The detector finds biphasic K-complex candidates. Use sleep-stage context, especially N2 scoring, when interpreting candidates in real data.

In [ ]:
k_signal=rng.normal(0,3,times.size)
for center in (8,25,44):
    k_signal+=-120*np.exp(-0.5*((times-center)/0.12)**2)
    k_signal+=80*np.exp(-0.5*((times-(center+0.45))/0.18)**2)
raw_kcomp=mne.io.RawArray((k_signal*1e-6)[None,:],info,verbose=False)
kcomplexes=detect_kcomp(raw_kcomp)
print('K-complex candidates:',kcomplexes['count'])
kcomplexes['events'][:2]